In [1]:
import requests
import zipfile
import io
import polars as pl
from datetime import datetime, timedelta
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
BASE_URL = "http://data.gdeltproject.org/gdeltv2"
DATA_DIR = Path("daily")
DATA_DIR.mkdir(exist_ok=True)

START_DATE = datetime(2020, 1, 1)

FINANCE_KEYWORDS = (
    "ECON_", "MARKET", "STOCK", "STOCKS", "SHARE", "SHARES",
    "BOND", "BONDS", "IPO", "EARNINGS",
    "INFLATION", "INTEREST_RATE", "CENTRAL_BANK",
    "MONETARY_POLICY", "GDP", "ECONOMY",
    "DIVIDEND", "MERCADO", "ACAO", "ACOES", "BOLSA",
    "JUROS", "TAXA_DE_JUROS",
    "INFLACAO", "BANCO_CENTRAL",
    "POLITICA_MONETARIA",
    "ECONOMIA", "PIB",
    "OFERTA_PUBLICA",
    "IBOVESPA", "IBOV11"
)

FINANCE_REGEX = f"({"|".join(FINANCE_KEYWORDS)})"

In [ ]:
def process_timestamp(timestamp: datetime):
    filename = timestamp.strftime("%Y%m%d%H%M%S") + ".gkg.csv.zip"
    url = f"{BASE_URL}/{filename}"

    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200:
            return None

        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open(z.namelist()[0]) as f:

                df = pl.scan_csv(
                    f,
                    separator="\t",
                    has_header=False,
                    columns=[1, 3, 4, 7, 8, 9, 15, 26],
                    new_columns=[
                        "datetime",
                        "source",
                        "url",
                        "themes",
                        "v2themes",
                        "locations",
                        "tone",
                        "extras"
                    ],
                    ignore_errors=True
                )

        # Brazil filter
        # brazil_mask = (
        #     pl.col("locations").str.to_lowercase().str.contains("brazil", literal=True) |
        #     pl.col("themes").str.to_lowercase().str.contains("brazil", literal=True) |
        #     pl.col("v2themes").str.to_lowercase().str.contains("brazil", literal=True)
        # )

        # Finance filter
        finance_mask = (
            pl.col("themes").str.to_uppercase().str.contains(FINANCE_REGEX) |
            pl.col("v2themes").str.to_uppercase().str.contains(FINANCE_REGEX)
        )

        # df = df.filter(brazil_mask & finance_mask)
        df = df.filter(finance_mask)

        if df.height == 0:
            return None

        df = df.with_columns([
            pl.col("tone")
                .cast(pl.Utf8)
                .str.split(",")
                .list.get(0)
                .cast(pl.Float32, strict=False)
                .alias("tone"),

            pl.col("datetime")
                .cast(pl.Utf8)
                .str.strptime(pl.Datetime, format="%Y%m%d%H%M%S", strict=False)
                .alias("datetime"),

            pl.col("extras")
                .cast(pl.Utf8)
                .str.extract(r"<PAGE_TITLE>(.*?)</PAGE_TITLE>", 1)
                .alias("title")
        ]).drop("extras")

        return df

    except Exception:
        return None

In [4]:
def process_date(date: datetime):
    timestamps = [
        date.replace(hour=h, minute=m)
        for h in range(24)
        for m in range(0, 60, 15)
    ]

    collected: list[pl.DataFrame] = []

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(process_timestamp, ts) for ts in timestamps]

        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                collected.append(result)

    if collected:
        final_df: pl.DataFrame = pl.concat(collected)
        final_df.write_parquet(DATA_DIR / f"{date.date()}.parquet")
        print("Saved", final_df.height, "rows for date", date.date())
    else:
        print("No Brazil finance news found.")


In [5]:
def process_time_frame(end_date: datetime):
    try:
        with open(DATA_DIR / "last_file.txt", "r") as file:
            last_file_downloaded = datetime.strptime(
                file.read().strip(), "%Y%m%d%H%M%S"
            )
    except:
        last_file_downloaded = START_DATE - timedelta(days=1)

    current_date = last_file_downloaded + timedelta(days=1)

    while current_date <= end_date:
        process_date(current_date)
        with open(DATA_DIR / "last_file.txt", "w") as file:
            file.write(current_date.strftime("%Y%m%d%H%M%S"))
        current_date += timedelta(days=1)

In [6]:
def concat_dfs(directory: Path):
    dfs = [pl.read_parquet(f) for f in directory.glob("*.parquet")]
    return pl.concat(dfs)

In [8]:
END_DATE = datetime(2026, 1, 1)
process_time_frame(END_DATE)

No Brazil finance news found.
